In [ ]:
"""
MF-DFA and MF-DMA on three series types: Prices, Log-Prices, Returns
=======================================================================

Key corrections vs. the original script
-----------------------------------------
1.  **No look-ahead (rolling) bias in MF-DMA.**
    The centred moving average used in MF-DMA must only use data that falls
    within the segment boundary.  The original code used a fixed global
    window around each *index*, which means points near the start / end of a
    segment peeked into neighbouring segments.  Here every detrending window
    is clipped to the segment it belongs to, so the trend at position k is
    always the average of x[v*s … (v+1)*s - 1], centred at k (or shifted by
    theta), with no cross-segment contamination.

2.  **MF-DFA integration step.**
    Classic MF-DFA integrates the series first (cumulative-sum of demeaned
    returns / values) before detrending.  The original skipped this entirely.
    Here:
      • Prices   → integrate (cumsum of demeaned prices), then detrend
      • Log-prices → same
      • Returns  → integrate (cumsum of demeaned returns = profile), then detrend
    This matches Kantelhardt et al. (2002) and is the standard definition.
    For MF-DMA, integration is NOT applied; the algorithm detrends the raw
    series directly with a moving average.

3.  **Segment completeness.**
    Only non-overlapping, complete segments of length s are used.  Remainder
    data at the tail (and optionally at the head for DMA) are discarded.

4.  **q = 0 handled correctly.**
    Geometric mean of rms² (or rms^q) is used for q = 0, consistent with
    L'Hôpital / log-average definition.

5.  **Positive RMS guard.**
    Zero-rms segments are dropped before computing F_q(s) to avoid
    log(0) or division by zero.

6.  **Scale range sanity.**
    scale_min is checked to be > polynomial order (1) + 1 = 2.
    scale_max is capped at N // 4 so every scale has at least 4 complete
    segments, giving a reliable least-squares fit.

7.  **Singularity spectrum regularisation.**
    np.gradient gives noisy α when h(q) is flat; a Savitzky-Golay smooth is
    applied before computing α and D(α).

8.  **No figure blocking.**
    plt.show() replaced by savefig so the script can run headless / in batch.
"""

import numpy as np
import yfinance as yf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
import os
import warnings

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# PARAMETERS
# ─────────────────────────────────────────────────────────────────────────────
TICKERS  = ["AAPL", "MSFT", "GOOG", "TSLA", "AMZN", "GC=F", "BTC-USD"]
START    = "2010-01-01"
END      = "2026-01-01"

Q_VALS   = np.arange(-10, 12, 2, dtype=float)   # float avoids integer mishaps
SCALE_MIN_REQ = 20       # absolute minimum scale requested
N_SCALES = 30
POLY_ORD = 1             # linear detrending in DFA

OUT_DIR  = "mf_plots"
os.makedirs(OUT_DIR, exist_ok=True)


# ─────────────────────────────────────────────────────────────────────────────
# SERIES PREPARATION
# ─────────────────────────────────────────────────────────────────────────────
def prepare_series(price: np.ndarray):
    """Return (prices, log_prices, returns) as clean 1-D arrays."""
    price      = price[price > 0]                           # drop non-positive
    log_price  = np.log(price)
    returns    = np.diff(log_price)                         # log-returns, length N-1
    return price, log_price, returns


def integrate(x: np.ndarray) -> np.ndarray:
    """Profile (integrated / cumsum of demeaned series). Used by MF-DFA."""
    return np.cumsum(x - x.mean())


# ─────────────────────────────────────────────────────────────────────────────
# MF-DFA  — with integration, no look-ahead bias
# ─────────────────────────────────────────────────────────────────────────────
def mfdfa(x: np.ndarray,
          q_vals: np.ndarray,
          scale_min: int,
          scale_max: int,
          n_scales: int,
          poly_ord: int = 1):
    """
    Standard MF-DFA (Kantelhardt et al. 2002).

    x         : raw series (prices / log-prices / returns).
                Integration (profile) is done internally.
    Returns   : scales, Fqs [len(q_vals) × len(scales)], hq [len(q_vals)]
    """
    # --- integrate ---
    y = integrate(x)
    N = len(y)

    scale_max = min(scale_max, N // 4)
    scale_min = max(scale_min, poly_ord + 2)

    scales = np.unique(
        np.logspace(np.log10(scale_min), np.log10(scale_max), n_scales)
        .astype(int)
    )
    scales = scales[scales >= scale_min]

    Fqs = np.full((len(q_vals), len(scales)), np.nan)

    for i, s in enumerate(scales):
        Ns = N // s            # number of complete non-overlapping segments
        if Ns < 4:
            continue

        rms = []
        for v in range(Ns):
            seg = y[v * s : (v + 1) * s]
            t   = np.arange(s)
            coef  = np.polyfit(t, seg, poly_ord)
            trend = np.polyval(coef, t)
            r = np.sqrt(np.mean((seg - trend) ** 2))
            if r > 0:
                rms.append(r)

        rms = np.asarray(rms)
        if len(rms) == 0:
            continue

        for j, q in enumerate(q_vals):
            if q == 0:
                Fqs[j, i] = np.exp(0.5 * np.mean(np.log(rms ** 2)))
            else:
                Fqs[j, i] = (np.mean(rms ** q)) ** (1.0 / q)

    # Compute h(q) via log-log regression, ignoring nan columns
    hq = np.full(len(q_vals), np.nan)
    for j in range(len(q_vals)):
        row  = Fqs[j]
        mask = np.isfinite(row) & (row > 0)
        if mask.sum() >= 4:
            hq[j] = np.polyfit(np.log(scales[mask]), np.log(row[mask]), 1)[0]

    return scales, Fqs, hq


# ─────────────────────────────────────────────────────────────────────────────
# MF-DMA  — segment-local moving average, no look-ahead
# ─────────────────────────────────────────────────────────────────────────────
def mfdma(x: np.ndarray,
          q_vals: np.ndarray,
          scale_min: int,
          scale_max: int,
          n_scales: int,
          theta: float = 0.5):
    """
    MF-DMA (Gu & Zhou 2010, Kristoufek 2013).

    No integration; the moving average IS the trend.
    theta = 0.0 → backward  (MF-BDMA)
    theta = 0.5 → centred   (MF-CDMA)
    theta = 1.0 → forward   (MF-FDMA)

    Bias fix
    --------
    The moving-average window at position k (within a segment starting at
    v*s) uses only data inside that segment:
        window = x[max(v*s, k - k_back) : min((v+1)*s, k + k_fwd + 1)]
    This prevents any cross-segment look-ahead or look-back.
    """
    N = len(x)
    scale_max = min(scale_max, N // 4)
    scale_min = max(scale_min, 4)

    scales = np.unique(
        np.logspace(np.log10(scale_min), np.log10(scale_max), n_scales)
        .astype(int)
    )
    scales = scales[scales >= scale_min]

    Fqs = np.full((len(q_vals), len(scales)), np.nan)

    for i, s in enumerate(scales):
        Ns = N // s
        if Ns < 4:
            continue

        k_back = int(np.floor((s - 1) * theta))
        k_fwd  = int(np.ceil ((s - 1) * (1.0 - theta)))

        # Collect squared residuals, one per (segment, valid position)
        sq_res_all = []

        for v in range(Ns):
            seg_start = v * s
            seg_end   = (v + 1) * s      # exclusive

            for k in range(seg_start, seg_end):
                # window clipped to current segment — NO cross-segment peeking
                w_start = max(seg_start, k - k_back)
                w_end   = min(seg_end,   k + k_fwd + 1)   # exclusive
                window  = x[w_start : w_end]

                if len(window) == 0:
                    continue

                trend = window.mean()
                sq_res_all.append((x[k] - trend) ** 2)

        sq_res = np.asarray(sq_res_all)
        sq_res = sq_res[sq_res > 0]

        if len(sq_res) == 0:
            continue

        for j, q in enumerate(q_vals):
            if q == 0:
                Fqs[j, i] = np.exp(0.5 * np.mean(np.log(sq_res)))
            else:
                Fqs[j, i] = (np.mean(sq_res ** (q / 2.0))) ** (1.0 / q)

    hq = np.full(len(q_vals), np.nan)
    for j in range(len(q_vals)):
        row  = Fqs[j]
        mask = np.isfinite(row) & (row > 0)
        if mask.sum() >= 4:
            hq[j] = np.polyfit(np.log(scales[mask]), np.log(row[mask]), 1)[0]

    return scales, Fqs, hq


# ─────────────────────────────────────────────────────────────────────────────
# MULTIFRACTAL SPECTRUM
# ─────────────────────────────────────────────────────────────────────────────
def multifractal_spectrum(q_vals: np.ndarray, hq: np.ndarray):
    """
    tau(q) = q * h(q) - 1
    alpha  = d tau / d q  (smoothed numerically)
    D(alpha) = q * alpha - tau(q)
    """
    tau_q = q_vals * hq - 1.0

    # Smooth tau before differentiation to reduce gradient noise
    if len(tau_q) >= 5:
        wl = min(5, len(tau_q) if len(tau_q) % 2 == 1 else len(tau_q) - 1)
        tau_smooth = savgol_filter(tau_q, window_length=wl, polyorder=2)
    else:
        tau_smooth = tau_q

    alpha   = np.gradient(tau_smooth, q_vals)
    D_alpha = q_vals * alpha - tau_smooth

    return tau_q, alpha, D_alpha


# ─────────────────────────────────────────────────────────────────────────────
# PLOTTING
# ─────────────────────────────────────────────────────────────────────────────
SERIES_LABELS = {
    "prices":      "Prices",
    "log_prices":  "Log-Prices",
    "returns":     "Log-Returns",
}

def plot_4panel(ticker, method, series_key, raw_series,
                scales, Fqs, hq, q_vals, tau_q, alpha, D_alpha):

    fig, axs = plt.subplots(2, 2, figsize=(13, 9))
    series_label = SERIES_LABELS[series_key]
    fig.suptitle(f"{method} · {ticker} · {series_label}", fontsize=14, fontweight="bold")

    # (a) series
    axs[0, 0].plot(raw_series, color="black", linewidth=0.7)
    axs[0, 0].set_title(f"(a) {series_label}")
    axs[0, 0].set_xlabel("Time index")
    axs[0, 0].set_ylabel(series_label)

    # (b) F_q(s)
    cmap = plt.cm.plasma
    colors = cmap(np.linspace(0, 1, len(q_vals)))
    for idx_q, q in enumerate(q_vals):
        row  = Fqs[idx_q]
        mask = np.isfinite(row) & (row > 0)
        if mask.sum() < 2:
            continue
        axs[0, 1].loglog(scales[mask], row[mask], 'o-',
                         color=colors[idx_q], markersize=3,
                         linewidth=1, label=f"q={int(q)}")
    axs[0, 1].set_title("(b) Fluctuation functions $F_q(s)$")
    axs[0, 1].set_xlabel("Scale $s$")
    axs[0, 1].set_ylabel("$F_q(s)$")
    axs[0, 1].legend(fontsize=7, ncol=2)

    # (c) h(q) and tau(q)
    valid = np.isfinite(hq)
    ax_h = axs[1, 0]
    ax_h.plot(q_vals[valid], hq[valid], 'o-', color="steelblue", label="$h(q)$")
    ax_h.set_title("(c) Generalized Hurst exponent & $\\tau(q)$")
    ax_h.set_xlabel("$q$")
    ax_h.set_ylabel("$h(q)$", color="steelblue")
    ax_h.tick_params(axis="y", labelcolor="steelblue")

    ax_tau = ax_h.twinx()
    ax_tau.plot(q_vals[valid], tau_q[valid], 's--', color="darkorange",
                label=r"$\tau(q)$")
    ax_tau.set_ylabel(r"$\tau(q)$", color="darkorange")
    ax_tau.tick_params(axis="y", labelcolor="darkorange")

    # (d) singularity spectrum
    valid_s = np.isfinite(alpha) & np.isfinite(D_alpha)
    axs[1, 1].plot(alpha[valid_s], D_alpha[valid_s], 'o-', color="crimson")
    axs[1, 1].set_title("(d) Singularity spectrum $D(\\alpha)$")
    axs[1, 1].set_xlabel(r"$\alpha$")
    axs[1, 1].set_ylabel(r"$D(\alpha)$")

    plt.tight_layout()
    fname = f"{OUT_DIR}/{ticker.replace('=','_')}_{method.replace('-','_')}_{series_key}.png"
    plt.savefig(fname, dpi=120, bbox_inches="tight")
    plt.close(fig)
    print(f"    saved → {fname}")


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
DMA_VARIANTS = {
    "MF-DFA":  None,      # theta unused for DFA
    "MF-BDMA": 0.0,
    "MF-CDMA": 0.5,
    "MF-FDMA": 1.0,
}

for ticker in TICKERS:
    print(f"\n{'='*60}")
    print(f"Ticker: {ticker}")

    data  = yf.download(ticker, start=START, end=END, progress=False)
    close = data["Close"].dropna().values.ravel().astype(float)

    if len(close) < 200:
        print("  Not enough data, skipping.")
        continue

    price, log_price, returns = prepare_series(close)

    series_dict = {
        "prices":     price,
        "log_prices": log_price,
        "returns":    returns,
    }

    for method, theta in DMA_VARIANTS.items():
        print(f"  Method: {method}")

        for series_key, x in series_dict.items():
            N = len(x)
            scale_max_eff = min(SCALE_MAX := 800, N // 4)
            scale_min_eff = max(SCALE_MIN_REQ, POLY_ORD + 2)

            print(f"    Series: {series_key}  (N={N}, "
                  f"scales [{scale_min_eff}, {scale_max_eff}])")

            if method == "MF-DFA":
                scales, Fqs, hq = mfdfa(
                    x, Q_VALS, scale_min_eff, scale_max_eff, N_SCALES, POLY_ORD
                )
            else:
                scales, Fqs, hq = mfdma(
                    x, Q_VALS, scale_min_eff, scale_max_eff, N_SCALES, theta
                )

            tau_q, alpha, D_alpha = multifractal_spectrum(Q_VALS, hq)

            plot_4panel(
                ticker, method, series_key, x,
                scales, Fqs, hq, Q_VALS,
                tau_q, alpha, D_alpha
            )

print("\nAll done.  Figures saved in:", OUT_DIR)